**What You'll Build**

*   Build an AI agent that takes a loan applicant's profile as input and produces a credit recommendation: Approve, Reject, or Refer to Human Review — along with a risk tier (Low / Medium / High), a score (0–100), and a plain-English explanation that could be shared with the applicant or a loan officer.

*   Your agent should work in at least two steps. First, a data assessment step: the agent examines the applicant's inputs, identifies gaps or red flags, and potentially asks clarifying questions (you can simulate this as a second-pass prompt). Second, a scoring and recommendation step: the agent applies a reasoning framework to produce the output. The key challenge here is designing the system prompt so the agent's reasoning is consistent, explainable, and defensible — not arbitrary. Think about what a credit committee would ask: can you stand behind this decision?

*   Design a fictional but realistic loan applicant schema: income, employment type, credit history, existing EMIs, loan amount requested, purpose of loan, tenure.
• Write a system prompt that gives the agent a credit policy: what counts as a strong applicant, what are automatic disqualifiers, what sends someone to human review?
• Build a minimum of 5 test applicants — a strong case, a weak case, a borderline case, a gig worker with irregular income, and a young applicant with no credit history.
• For each applicant, show the agent's reasoning trace — not just the final score.
• Stretch goal: build a second agent that acts as a 'credit committee reviewer' and challenges the first agent's recommendation.



# **Model Development Agent**

In [16]:
!pip install langchain_openai
!pip install langchain-google-genai
!pip install langchain-groq

In [18]:
# Set your OpenAI API key as an environment variable
import os
import pandas as pd
import json
os.environ["GROQ_API_KEY"] = "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"
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",   # Best balance of quality + speed
    temperature=0
)


Step 1 - Ideal Applicant Schema or Loan Application

In [19]:
#Define an ideal applicant schema
from pydantic import BaseModel
from typing import Optional

class Applicant(BaseModel):
    applicant_id: str
    age: int
    income: float  # monthly
    employment_type: str  # salaried, self-employed, gig
    employment_tenure: float  # in years
    credit_score: Optional[int]
    existing_emi: float
    loan_amount: float
    loan_tenure: int  # months
    loan_purpose: str

Step 2 - Credit Policy (CORE of your system)

CREDIT POLICY:

Strong Applicant:
- Credit score > 750
- DTI (EMI/income) < 35%
- Stable employment (>2 years salaried)
- Low existing obligations

Moderate:
- Credit score 650–750
- DTI 35–50%
- Some instability or thin file

Weak:
- Credit score < 650
- DTI > 50%
- Irregular or unverifiable income

Automatic Reject:
- Credit score < 550
- Missing income
- EMI > income

Refer to Human:
- No credit history
- Gig workers with high income variability
- Borderline DTI (45–55%)
- Inconsistent profile

Scoring Guide (0–100):
- Credit score: 40%
- DTI: 30%
- Employment stability: 20%
- Profile quality: 10%

3. Step 1 Agent — Data Assessment

In [20]:
def data_assessment(applicant_json):
    prompt = f"""
    You are a credit pre-screening agent.

    Analyze this applicant:
    {applicant_json}

    Identify:
    - Missing fields among only the defined set of attributes under each application.
    Don't halucinate and choose only from the defined set of attributes, so if let's suppose among 10 attributes one i.e., credit_score is missing, then only show that as missing and higlight that.
    - Red flags if any, among only the defined set of attributes under each application.
    - Clarifying questions, if any, among only the defined set of attributes under each application.

    Output JSON:
    {{
      "missing_fields": "",
      "red_flags": "",
      "questions": ""
    }}
    """

    response = llm.invoke(prompt)
    return response.content


4. Step 2 Agent — Scoring + Decision

In [21]:
def scoring_agent(applicant_json, policy):
    prompt = f"""
    You are a credit assessment agent.

    Follow policy:
    {policy}

    Assess each Applicant, basis the policy defined above:
    {applicant_json}

    IMPORTANT:
    - Return ONLY valid JSON
    - Do NOT add explanations outside JSON
    - Do NOT use markdown
    - score cannot be zero, it should have a value
    - restrict the risk tier between Low, Medium, High
    - recommendation can be Approve, Reject, or Refer to Human

    Output JSON:
    {{
      "applicant_id": "",
      "score": "",
      "risk_tier": "",
      "recommendation": "",
      "reasoning_trace": "",
      "explanation": ""
    }}

    You MUST return ONLY valid JSON.

    Include ALL fields with all guardrails mentioned above in the IMPORTANT writeup:
    - score (Mandatory & Non Zero)
    - risk_tier (Mandatory)
    - recommendation (Mandatory and should be between Approve, Reject, or Refer to Human)
    - reasoning_trace
    - explanation

    If any field is missing, infer it explicitly. Do NOT omit fields.
    Do NOT write text outside JSON.
    Do NOT use markdown.

    Output:
    {{
      "applicant_id": "",
      "score": "",
      "risk_tier": "",
      "recommendation": "",
      "reasoning_trace": "",
      "explanation": ""
    }}
    """

    response = llm.invoke(prompt)
    return response.content

5. Second Agent — Credit Committee Reviewer

In [22]:
def reviewer_agent(decision_output):
    prompt = f"""
    You are a credit committee reviewer.

    Review this decision:
    {decision_output}

    Challenge:
    - Is the decision justified?
    - Any overlooked risks?
    - Should it be escalated?

    Output:
    {{
      "agreement": "Yes/No",
      "concerns": "",
      "final_decision": ""
    }}
    """

    response = llm.invoke(prompt)
    return response.content


6. Test Applicants

In [23]:
test_applicants = [
{
 "applicant_id": "A1",
 "age": 35,
 "income": 25000,
 "employment_type": "salaried",
 "employment_tenure": 5,
 "credit_score": 780,
 "existing_emi": 5000,
 "loan_amount": 200000,
 "loan_tenure": 36,
 "loan_purpose": "car"
},

{
 "applicant_id": "A2",
 "age": 40,
 "income": 8000,
 "employment_type": "gig",
 "employment_tenure": 1,
 "credit_score": 600,
 "existing_emi": 5000,
 "loan_amount": 150000,
 "loan_tenure": 24,
 "loan_purpose": "personal"
},

{
 "applicant_id": "A3",
 "age": 29,
 "income": 12000,
 "employment_type": "salaried",
 "employment_tenure": 1.5,
 "credit_score": 680,
 "existing_emi": 4000,
 "loan_amount": 100000,
 "loan_tenure": 24,
 "loan_purpose": "education"
},

{
 "applicant_id": "A4",
 "age": 32,
 "income": 20000,
 "employment_type": "gig",
 "employment_tenure": 3,
 "credit_score": None,
 "existing_emi": 3000,
 "loan_amount": 120000,
 "loan_tenure": 36,
 "loan_purpose": "business"
},

{
 "applicant_id": "A5",
 "age": 23,
 "income": 9000,
 "employment_type": "salaried",
 "employment_tenure": 0.5,
 "credit_score": None,
 "existing_emi": 0,
 "loan_amount": 50000,
 "loan_tenure": 12,
 "loan_purpose": "consumer"
}
]

7. Run Full Workflow

In [24]:
policy = """CREDIT POLICY:

Strong Applicant:
- Credit score > 750
- DTI (EMI/income) < 35%
- Stable employment (>2 years salaried)
- Low existing obligations

Moderate:
- Credit score 650-750
- DTI 35-50%
- Some instability or thin file

Weak:
- Credit score < 650
- DTI > 50%
- Irregular or unverifiable income

Automatic Reject:
- Credit score < 550
- Missing income
- EMI > income

Refer to Human:
- No credit history
- Gig workers with high income variability
- Borderline DTI (45-55%)
- Inconsistent profile

Scoring Guide (0-100):
- Credit score: 40%
- DTI: 30%
- Employment stability: 20%
- Profile quality: 10%"""

results = []

for applicant in test_applicants:
    assessment = data_assessment(applicant)
    decision = scoring_agent(applicant, policy)
    review = reviewer_agent(decision)

    results.append({
        "input": applicant,
        "assessment": assessment,
        "decision": decision,
        "review": review
    })


In [25]:
def safe_parse(response):
    if isinstance(response, dict):
        return response

    if response is None:
        return {}

    text = str(response)

    # extract last JSON block (most important fix)
    blocks = re.findall(r"```(?:json)?\s*([\s\S]*?)```", text)

    for block in reversed(blocks):
        try:
            return json.loads(block.strip())
        except:
            continue

    # fallback: try last {...}
    start = text.rfind("{")
    end = text.rfind("}")

    if start != -1 and end != -1:
        try:
            return json.loads(text[start:end+1])
        except:
            pass

    return {}

In [27]:
import re

rows = []

for r in results:
    applicant = r["input"]

    assessment = safe_parse(r["assessment"])
    decision = safe_parse(r["decision"])
    review = safe_parse(r["review"])

    rows.append({
        "applicant_id": applicant.get("applicant_id"),
        "income": applicant.get("income"),
        "employment_type": applicant.get("employment_type"),
        "credit_score": applicant.get("credit_score"),
        "existing_emi": applicant.get("existing_emi"),
        "loan_amount": applicant.get("loan_amount"),

        "missing_fields": assessment.get("missing_fields"),
        "red_flags": assessment.get("red_flags"),

        "score": decision.get("score"),
        "risk_tier": decision.get("risk_tier"),
        "recommendation": decision.get("recommendation"),
        "reasoning_trace": decision.get("reasoning_trace"),
        "explanation": decision.get("explanation"),

        "review_agreement": review.get("agreement"),
        "review_concerns": review.get("concerns"),
        "final_decision": review.get("final_decision"),
    })

df = pd.DataFrame(rows)

print(df)

  applicant_id  income employment_type  credit_score  existing_emi  \
0           A1   25000        salaried         780.0          5000   
1           A2    8000             gig         600.0          5000   
2           A3   12000        salaried         680.0          4000   
3           A4   20000             gig           NaN          3000   
4           A5    9000        salaried           NaN             0   

   loan_amount missing_fields  \
0       200000                  
1       150000                  
2       100000                  
3       120000   credit_score   
4        50000   credit_score   

                                           red_flags score risk_tier  \
0        High debt-to-income ratio with the new loan    85       Low   
1  Unstable employment type, short employment ten...    42      High   
2  Short employment tenure, high existing EMI com...    64    Medium   
3  employment_type is 'gig', employment_tenure is...    20      High   
4    short employmen

In [28]:
print(type(r["assessment"]))
print(r["assessment"])
print("-" * 80)

<class 'str'>
Based on the provided applicant data, here's the analysis:

The applicant data is: 
{'applicant_id': 'A5', 'age': 23, 'income': 9000, 'employment_type': 'salaried', 'employment_tenure': 0.5, 'credit_score': None, 'existing_emi': 0, 'loan_amount': 50000, 'loan_tenure': 12, 'loan_purpose': 'consumer'}

**Missing fields:**
The 'credit_score' field is missing.

**Red flags:**
The 'employment_tenure' is 0.5 years, which is relatively short. The 'income' is 9000, which might be low considering the 'loan_amount' of 50000.

**Clarifying questions:**
What is the basis for the 'income' of 9000, is it monthly or yearly? How does the applicant plan to repay the loan with a relatively short 'employment_tenure'?

Output JSON:
```
{
  "missing_fields": "credit_score",
  "red_flags": "short employment tenure, potentially low income",
  "questions": "basis for income, plan for loan repayment"
}
```
--------------------------------------------------------------------------------


In [29]:
df.to_csv("credit_decision_output.csv", index=False)
from google.colab import files
files.download("credit_decision_output.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>